# 03 — Point-to-Point Navigation

Navigate through a sequence of waypoints using Nav2 path planning.

**What you'll learn**:
- Single-goal navigation with `move_to()`
- Multi-waypoint paths with `follow_path()`
- Handling navigation failures gracefully

**Prerequisites**: `pip install threewe[sim]`

In [ ]:
import asyncio
from threewe import Robot
from threewe.types import Pose2D

In [ ]:
robot = Robot(backend="gazebo", scene="office_v2")
robot.connect()
print(f"Start pose: {robot.get_pose()}")

## Single Goal Navigation

`move_to(x, y)` plans a collision-free path and executes it.
Returns a `MoveResult` with success status, distance, and duration.

In [ ]:
# Navigate to a point in the office
result = await robot.move_to(x=5.0, y=3.0)
print(f"Goal 1: success={result.success}, reason={result.reason}")
print(f"  Traveled {result.distance:.2f}m in {result.duration:.1f}s")

## Navigation with Orientation

Optionally specify a final heading angle (radians).

In [ ]:
import math

# Navigate and face north (theta=pi/2)
result = await robot.move_to(x=8.0, y=7.0, theta=math.pi / 2)
print(f"Goal 2: success={result.success}")
print(f"  Final pose: {robot.get_pose()}")

## Multi-Waypoint Path

`follow_path()` navigates through a list of waypoints in sequence.
Useful for patrol routes, coverage patterns, or guided tours.

In [ ]:
# Define a patrol route through the office
waypoints = [
    Pose2D(x=2.0, y=2.0, theta=0.0),
    Pose2D(x=6.0, y=2.0, theta=0.0),
    Pose2D(x=6.0, y=8.0, theta=math.pi / 2),
    Pose2D(x=2.0, y=8.0, theta=math.pi),
    Pose2D(x=2.0, y=2.0, theta=-math.pi / 2),
]

print(f"Following {len(waypoints)} waypoints...")
result = await robot.follow_path(waypoints)
print(f"Path complete: success={result.success}")
print(f"  Total distance: {result.distance:.2f}m")
print(f"  Total time: {result.duration:.1f}s")

## Handling Failures

Navigation can fail (blocked path, unreachable goal). Always check `result.success`.

In [ ]:
from threewe.exceptions import NavigationError

# Try navigating to a point that might be inside a wall
result = await robot.move_to(x=0.0, y=0.0, timeout=10.0)
if not result.success:
    print(f"Navigation failed: {result.reason}")
    print(f"Robot stopped at: {result.final_pose}")
else:
    print("Reached goal!")

## Simple Motion Primitives

For quick movements without full path planning:

In [ ]:
# Drive forward 1 meter
result = await robot.move_forward(1.0)
print(f"Forward 1m: {result.reason}")

# Rotate 90 degrees counter-clockwise
result = await robot.rotate(math.pi / 2)
print(f"Rotate 90°: {result.reason}")

print(f"Final pose: {robot.get_pose()}")

In [ ]:
robot.disconnect()